In [ ]:
!pip install unsloth
!pip install gradio
!pip install bitsandbytes
!pip install accelerate


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.5/61.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.8/348.8 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.5/511.5 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.7/276.7 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 15.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      

In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/unsloth_law_model_lora.zip"  # 코랩에 복사한 위치
extract_path = "/content/unsloth_law_model_lora2"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = extract_path,  # 압축해제한 경로
    max_seq_length = 2048,
    dtype = torch.float16,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model)  # Inference용 세팅


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.1: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Unsloth 2025.11.1 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
       

In [ ]:
import gradio as gr
import torch
import re
from difflib import SequenceMatcher

# === 유사 문장 판단 ===
def is_similar(a, b, threshold=0.85):
    return SequenceMatcher(None, a, b).ratio() > threshold

# === 문장 정제 ===
def clean_sentence(sentence):
    sentence = sentence.strip()
    sentence = re.sub(r'(\b\w+\b)(\s*에 대한\s*\1)+', r'\1', sentence)
    sentence = re.sub(r'\b(\w+)\s+\1\b', r'\1', sentence)
    if len(sentence) < 6 or sentence.endswith(("의", "에", "는", "은", "가", "로", "과", ",")):
        return None
    return sentence

# === 문장 끝마다 줄바꿈 적용 ===
def format_sentences_with_linebreaks(text):
    text = re.sub(r'\s+', ' ', text.strip())
    sentences = re.split(r'(?<=[.!?])\s+', text)
    # 중복 제거 후 문장 정제
    cleaned_sentences = []
    for s in sentences:
        cleaned = clean_sentence(s)
        if cleaned and not any(is_similar(cleaned, c) for c in cleaned_sentences):
            cleaned_sentences.append(cleaned)
    return "\n".join(cleaned_sentences)

# === 중단 문장 잘라내기 ===
def truncate_incomplete_sentences(text):
    if not re.search(r'[.!?]"?$', text.strip()):
        last_period = text.strip().rfind(".")
        if last_period != -1:
            return text[:last_period+1]
    return text

# === GPT 스타일 상담 함수 ===
def law_consultation_gpt(user_message, chat_history):
    prompt = f"[사용자]: {user_message}\n[AI 법률상담사]:"

    # 모델 토크나이즈 및 생성
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=800,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.05
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "[AI 법률상담사]:" in response:
        response = response.split("[AI 법률상담사]:")[-1].strip()
    response = truncate_incomplete_sentences(response)
    response = format_sentences_with_linebreaks(response)

    chat_history = chat_history or []
    chat_history.append((user_message, response))
    return chat_history, chat_history

# === Gradio Blocks UI ===
with gr.Blocks(theme=gr.themes.Soft(), css="""
    body { background-color: #f8f9fa; font-family: 'Noto Sans KR', sans-serif; }
    .navbar { background-color: #2C3E50; padding: 12px 24px; color: white; display: flex; justify-content: space-between; align-items: center; }
    .hero { text-align: center; padding: 40px 20px; background-color: #ECF0F1; border-bottom: 1px solid #ccc; }
    .hero h1 { font-size: 36px; color: #2C3E50; margin-bottom: 10px; }
    .hero p { font-size: 18px; color: #555; }
    .card { background: white; padding: 20px; border-radius: 12px; box-shadow: 0 4px 10px rgba(0,0,0,0.1); }
    .footer { text-align: center; margin-top: 30px; padding: 15px; font-size: 14px; color: gray; border-top: 1px solid #ccc; }

    /* Chatbot 말풍선 구분 */
    .gradio-chatbot-message.user { background-color: #3498db; color: white; border-radius: 20px 20px 0px 20px; padding: 10px; margin: 5px; }
    .gradio-chatbot-message.bot { background-color: #ecf0f1; color: #2c3e50; border-radius: 20px 20px 20px 0px; padding: 10px; margin: 5px; white-space: pre-wrap; }
""") as demo:

    # 네비게이션 바
    with gr.Row():
        gr.HTML("""
        <div class="navbar">
            <div style="font-size:20px; font-weight:bold; color:white;">⚖️ AI 법률상담소</div>
        </div>
        """)

    # Hero 영역
    with gr.Row():
        gr.HTML("""
        <div class="hero">
            <h1>AI 변호사 상담 시스템</h1>
            <p>LoRA로 학습된 LLaMA 모델 기반의 법률 상담 서비스입니다.<br>
            명확하고 신뢰할 수 있는 답변을 제공합니다.</p>
        </div>
        """)

    # 메인 GPT 스타일 채팅 영역
    with gr.Row():
        with gr.Column(scale=1):
            with gr.Group(elem_classes="card"):
                user_input = gr.Textbox(
                    label="법률 질문 입력",
                    placeholder="예: 이혼 시 재산 분할 기준은 어떻게 되나요?",
                    lines=4
                )
                submit_btn = gr.Button("전송", variant="primary")

        with gr.Column(scale=2):
            with gr.Group(elem_classes="card"):
                chat_history = gr.Chatbot(label="AI 법률상담사", elem_id="chatbot")

    # 푸터
    with gr.Row():
        gr.HTML("""
        <div class="footer">
            © 2025 AI 법률상담소 | 본 상담은 참고용이며, 실제 법률 자문은 변호사와 상의하세요.
        </div>
        """)

    # 이벤트 연결
    submit_btn.click(
        fn=law_consultation_gpt,
        inputs=[user_input, chat_history],
        outputs=[chat_history, chat_history]
    )

demo.launch(share=True)


/tmp/ipython-input-2068001817.py:108: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chat_history = gr.Chatbot(label="AI 법률상담사", elem_id="chatbot")


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://66a5273541f800968d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
